In [4]:
import pandas as pd

# Scrape per-game stats
per_game_url = 'https://www.basketball-reference.com/leagues/NBA_2024_per_game.html'
per_game = pd.read_html(per_game_url)[0]
per_game = per_game[per_game['Rk'] != 'Rk']
per_game.reset_index(drop=True, inplace=True)

# Scrape advanced stats
advanced_url = 'https://www.basketball-reference.com/leagues/NBA_2024_advanced.html'
advanced = pd.read_html(advanced_url)[0]
advanced = advanced[advanced['Rk'] != 'Rk']
advanced.reset_index(drop=True, inplace=True)

# Clean column names
per_game.columns = per_game.columns.str.strip()
advanced.columns = advanced.columns.str.strip()

# Rename columns if necessary
if 'Team' in per_game.columns:
    per_game.rename(columns={'Team': 'Tm'}, inplace=True)
if 'Team' in advanced.columns:
    advanced.rename(columns={'Team': 'Tm'}, inplace=True)

# Merge on Player only (not on 'Tm') to avoid dropped rows
merged_stats = pd.merge(per_game, advanced, on='Player', suffixes=('_per_game', '_adv'))

# Optional: convert all numeric columns (starting from col 2 to skip Player)
for col in merged_stats.columns[1:]:
    merged_stats[col] = pd.to_numeric(merged_stats[col], errors='ignore')

# Preview merged data
merged_stats.head()


C:\Users\mikey\AppData\Local\Temp\ipykernel_22028\2827805392.py:30: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  merged_stats[col] = pd.to_numeric(merged_stats[col], errors='ignore')


,Rk_per_game,Player,Age_per_game,Tm_per_game,Pos_per_game,G_per_game,GS_per_game,MP_per_game,FG,FGA,...,USG%,OWS,DWS,WS,WS/48,OBPM,DBPM,BPM,VORP,Awards_adv
0,1.0,Joel Embiid,29.0,PHI,C,39.0,39.0,33.6,11.5,21.8,...,39.6,5.2,2.3,7.5,0.275,8.5,3.1,11.6,4.5,AS
1,2.0,Luka Dončić,24.0,DAL,PG,70.0,70.0,37.5,11.5,23.6,...,36.0,8.5,3.5,12.0,0.220,8.3,1.7,9.9,8.0,"MVP-3,CPOY-6,AS,NBA1"
2,3.0,Giannis Antetokounmpo,29.0,MIL,PF,73.0,73.0,35.2,11.5,18.8,...,33.0,9.5,3.7,13.2,0.246,6.7,2.4,9.0,7.2,"MVP-4,DPOY-9,CPOY-12,AS,NBA1"
3,4.0,Shai Gilgeous-Alexander,25.0,OKC,PG,75.0,75.0,34.0,10.6,19.8,...,32.8,10.5,4.2,14.6,0.275,6.7,2.3,9.0,7.1,"MVP-2,DPOY-7,CPOY-3,AS,NBA1"
4,5.0,Jalen Brunson,27.0,NYK,PG,77.0,77.0,35.4,10.3,21.4,...,32.5,8.8,2.4,11.2,0.198,6.3,-0.4,5.8,5.4,"MVP-5,CPOY-5,AS,NBA2"


In [5]:
# Scrape NBA 2023–24 salary data from HoopsHype
import pandas as pd

salary_url = "https://hoopshype.com/salaries/players/2023-2024/"
salary_tables = pd.read_html(salary_url)

# Extract the main salary table
salary_df = salary_tables[0]
salary_df.columns = ['Rank', 'Player', 'Team', 'Salary']  # Rename for clarity
salary_df.drop(columns='Rank', inplace=True)

# Clean up names: remove special characters like accents and extra whitespace
salary_df['Player'] = salary_df['Player'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
salary_df['Player'] = salary_df['Player'].str.strip()

# Clean salary column: remove $ and commas, convert to float
salary_df['Salary'] = salary_df['Salary'].replace('[\$,]', '', regex=True).astype(float)

# Preview
salary_df.head()


<>:17: SyntaxWarning: invalid escape sequence '\$'
<>:17: SyntaxWarning: invalid escape sequence '\$'
C:\Users\mikey\AppData\Local\Temp\ipykernel_22028\2057716035.py:17: SyntaxWarning: invalid escape sequence '\$'
  salary_df['Salary'] = salary_df['Salary'].replace('[\$,]', '', regex=True).astype(float)


,Player,Team,Salary
0,Stephen Curry,"$51,915,615",53458234.0
1,Kevin Durant,"$47,649,433",49065286.0
2,Nikola Jokic,"$47,607,350",49021953.0
3,LeBron James,"$47,607,350",49021953.0
4,Joel Embiid,"$47,607,350",49021953.0


In [8]:
import pandas as pd
import re

# Step 1: Define function to remove suffixes like Jr., II, III
def clean_player_name(name):
    name = re.sub(r'\s+Jr\.?', '', name)
    name = re.sub(r'\s+III', '', name)
    name = re.sub(r'\s+II', '', name)
    name = name.strip()
    return name

# Step 2: Apply cleaning function to both DataFrames
merged_stats['Player_clean'] = merged_stats['Player'].apply(clean_player_name)
salary_df['Player_clean'] = salary_df['Player'].apply(clean_player_name)

# Step 3: Merge using cleaned player names
final_df = pd.merge(merged_stats, salary_df, on='Player_clean', how='left', suffixes=('', '_salary'))

# Step 4: Check how many players are still missing salary info
missing_salary_count = final_df['Salary'].isna().sum()
print(f"❗ Players still missing salary info after name cleaning: {missing_salary_count}")

# Step 5: Preview merged result
print("\n✅ Sample merged data:")
print(final_df[['Player', 'Tm_per_game', 'Pos_per_game', 'Salary']].head(10))



❗ Players still missing salary info after name cleaning: 79

✅ Sample merged data:
                    Player Tm_per_game Pos_per_game      Salary
0              Joel Embiid         PHI            C  49021953.0
1              Luka Doncic         DAL           PG  41254687.0
2    Giannis Antetokounmpo         MIL           PF  46996232.0
3  Shai Gilgeous-Alexander         OKC           PG  34378905.0
4            Jalen Brunson         NYK           PG  27129530.0
5             Devin Booker         PHO           PG  37086384.0
6             Kevin Durant         PHO           PF  49065286.0
7             Jayson Tatum         BOS           PF  33568737.0
8             De'Aaron Fox         SAC           PG  33568737.0
9         Donovan Mitchell         CLE           SG  34147405.0


In [9]:
# Step 1: Choose modeling features (you can adjust this list)
features = [
    'PTS', 'AST', 'TRB', 'STL', 'BLK', 'MP_per_game',
    'TS%', 'BPM', 'VORP', 'WS', 'WS/48',
    'Age_per_game'
]

# Step 2: Drop rows with missing features or salary
model_df = final_df[features + ['Salary']].dropna()

# Step 3: Separate into features (X) and target (y)
X = model_df[features]
y = model_df['Salary']

# Step 4: Confirm shape
print(f"✅ Final modeling dataset: {X.shape[0]} players, {X.shape[1]} features")


✅ Final modeling dataset: 1151 players, 12 features


In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Step 1: Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Step 3: Predict on test set
y_pred = model.predict(X_test)

# Step 4: Evaluate model
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"📊 Linear Regression Results:")
print(f"➡️ RMSE: ${rmse:,.2f}")
print(f"➡️ R² Score: {r2:.4f}")


📊 Linear Regression Results:
➡️ RMSE: $6,633,734.01
➡️ R² Score: 0.5878


In [18]:
# Step 1: Refit model on all data
model.fit(X, y)
predicted_salary = model.predict(X)

# Step 2: Add predictions back to the DataFrame
model_df = model_df.copy()
model_df['PredictedSalary'] = predicted_salary
model_df['SalaryGap'] = model_df['PredictedSalary'] - model_df['Salary']
model_df['Player'] = final_df.loc[model_df.index, 'Player']

# Step 3: Convert all salary values to millions and round
formatted_df = model_df.copy()
formatted_df['Salary'] = (formatted_df['Salary'] / 1_000_000).round(2)
formatted_df['PredictedSalary'] = (formatted_df['PredictedSalary'] / 1_000_000).round(2)
formatted_df['SalaryGap'] = (formatted_df['SalaryGap'] / 1_000_000).round(2)

# Step 4: Sort and display results
print("🟢 Top 10 Most Underpaid Players (Predicted - Actual) [in Millions]:")
print(
    formatted_df.sort_values('SalaryGap', ascending=False)[
        ['Player', 'Salary', 'PredictedSalary', 'SalaryGap']
    ].head(10).to_string(index=False)
)

print("\n🔴 Top 10 Most Overpaid Players (Predicted - Actual) [in Millions]:")
print(
    formatted_df.sort_values('SalaryGap')[
        ['Player', 'Salary', 'PredictedSalary', 'SalaryGap']
    ].head(10).to_string(index=False)
)




🟢 Top 10 Most Underpaid Players (Predicted - Actual) [in Millions]:
           Player  Salary  PredictedSalary  SalaryGap
     Tyrese Maxey    4.47            29.15      24.67
     Desmond Bane    3.96            24.43      20.47
Tyrese Haliburton    5.98            25.37      19.39
       Cam Thomas    2.31            20.01      17.70
   Alperen Sengun    3.64            21.01      17.37
      Eric Gordon    3.29            19.25      15.96
   Jalen Williams    4.69            20.59      15.90
Immanuel Quickley    4.30            19.98      15.69
Russell Westbrook    3.95            19.38      15.43
  Kelly Oubre Jr.    2.98            17.49      14.51

🔴 Top 10 Most Overpaid Players (Predicted - Actual) [in Millions]:
        Player  Salary  PredictedSalary  SalaryGap
   Ben Simmons   39.02             6.44     -32.58
  Bradley Beal   48.13            23.38     -24.75
Gordon Hayward   34.31            10.84     -23.47
Gordon Hayward   34.31            10.93     -23.38
Gordon Hayward 

In [17]:
import plotly.express as px
import pandas as pd

# Prepare DataFrame with salaries in millions
plot_df = model_df.copy()
plot_df['ActualSalary_M'] = plot_df['Salary'] / 1_000_000
plot_df['PredictedSalary_M'] = plot_df['PredictedSalary'] / 1_000_000

# Create interactive scatter plot
fig = px.scatter(
    plot_df,
    x='ActualSalary_M',
    y='PredictedSalary_M',
    hover_name='Player',
    hover_data={
        'ActualSalary_M': ':.2f',
        'PredictedSalary_M': ':.2f',
    },
    title='📈 Interactive NBA Salary Prediction (2023–24)',
    labels={
        'ActualSalary_M': 'Actual Salary ($M)',
        'PredictedSalary_M': 'Predicted Salary ($M)'
    },
    opacity=0.6
)

# Add perfect prediction line
fig.add_shape(
    type='line',
    x0=plot_df['ActualSalary_M'].min(),
    y0=plot_df['ActualSalary_M'].min(),
    x1=plot_df['ActualSalary_M'].max(),
    y1=plot_df['ActualSalary_M'].max(),
    line=dict(color='red', dash='dash'),
    name='Perfect Prediction'
)

# Final layout tweaks
fig.update_layout(
    width=800,
    height=800,
    showlegend=False
)

fig.show()


In [21]:
# 💸 Enter your budget in millions
budget_millions = 1  # Change this number to adjust your budget

# Step 1: Filter players within the salary budget
value_pool = formatted_df[formatted_df['Salary'] <= budget_millions]

# Step 2: Sort by highest value gap (Predicted - Actual)
best_value = value_pool.sort_values('SalaryGap', ascending=False)

# Step 3: Display top value players under the budget
print(f"💰 Top Underpaid Players Within ${budget_millions:,}M Budget:")
print(
    best_value[['Player', 'Salary', 'PredictedSalary', 'SalaryGap']].head(10).to_string(index=False)
)


💰 Top Underpaid Players Within $1M Budget:
        Player  Salary  PredictedSalary  SalaryGap
 Javonte Green    0.27            13.78      13.51
   Danny Green    0.21             7.99       7.78
 Isaiah Thomas    0.49             8.11       7.62
Zavier Simpson    0.21             5.74       5.53
  RaiQuan Gray    0.15             5.19       5.05
 Jaylen Nowell    0.41             5.39       4.98
 Jaylen Nowell    0.41             5.35       4.94
 Jaylen Nowell    0.41             5.12       4.71
  Kobi Simmons    0.12             4.79       4.67
Mamadi Diakite    0.73             5.38       4.64
